# Fine-Tuning do BERT para Análise de Sentimentos (SST-2)
Este notebook replica o fine-tuning do modelo `BERT BASE` no dataset SST-2 (Stanford Sentiment Treebank), uma tarefa de classificação binária de uma única frase extraída de críticas de cinema.

In [1]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments
from datasets import load_dataset
import evaluate
import numpy as np

/home/tiduswr/miniconda3/envs/bert_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Verificação de Hardware
Esta célula verifica a disponibilidade da GPU para acelerar o processo.

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Rodando em: {torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU'}")

Rodando em: NVIDIA GeForce RTX 4060 Ti


# Dataset
Usando o SST-2 (citado na Tabela 1 do artigo)

In [3]:
dataset = load_dataset("nyu-mll/glue", "sst2")

# Configurações do Modelo

In [4]:
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)

def tokenize_fn(batch):
    return tokenizer(batch["sentence"], padding="max_length", truncation=True, max_length=128)

tokenized_datasets = dataset.map(tokenize_fn, batched=True)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11234.19it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpo

# Carrega a métrica de acurácia

In [5]:
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # Pega a classe com maior probabilidade
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

## 3. Configuração de Hiperparâmetros

In [6]:
training_args = TrainingArguments(
    output_dir="./.results",
    learning_rate=2e-5,                     # Uma das taxas de aprendizado testadas no artigo (pág. 6)
    per_device_train_batch_size=32,         # Batch size recomendado pelos autores (pág. 6)
    num_train_epochs=3,                     # Quantidade de épocas usada no artigo (pág. 6)
    fp16=True,                              # Tensor Cores pra 4060 Ti para acelerar
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

## 4. Execução do Treinamento
O fine-tuning do BERT é relativamente barato do ponto de vista computacional. Os autores relatam que todos os resultados podem ser replicados em algumas horas utilizando uma única GPU.

In [7]:
print("Iniciando a replicação do BERT...")
trainer.train()

Iniciando a replicação do BERT...


Epoch,Training Loss,Validation Loss,Accuracy
1,0.154122,0.243296,0.930046
2,0.094399,0.252107,0.933486
3,0.075059,0.287196,0.930046


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.05it/s]


TrainOutput(global_step=6315, training_loss=0.12863779966546163, metrics={'train_runtime': 928.0298, 'train_samples_per_second': 217.716, 'train_steps_per_second': 6.805, 'total_flos': 1.329019985058048e+16, 'train_loss': 0.12863779966546163, 'epoch': 3.0})